# 7.2 - Pesquisa vetorial [runs]

## 1. Parâmetros

In [6]:
PASTA_DOCS_QRELS = './dados/outputs/0 - qrel - docs - query - raw_human_eval/'

ARQUIVO_QUERIES = f'{PASTA_DOCS_QRELS}/query.csv'
ARQUIVO_QRELS = f'{PASTA_DOCS_QRELS}/qrel.csv'

PASTA_CADERNO_EMBEDDINGS = './dados/outputs/6 - embeddings/'

PASTA_RESULTADO_CADERNO = './dados/outputs/7 - runs/'

NOME_MODELO_EMB_OPENAI_LARGE = "text-embedding-3-large"
NOME_MODELO_EMB_OPENAI_SMALL = "text-embedding-3-small"
NOME_MODELO_EMB_GEMINI_768 = "gemini-embedding-001"

MODELOS_DISPONIVEIS = [NOME_MODELO_EMB_OPENAI_LARGE,
                       NOME_MODELO_EMB_OPENAI_SMALL,
                      NOME_MODELO_EMB_GEMINI_768]

## 2. Carrega as queries e o qrels

In [2]:
import pandas as pd

queries = pd.read_csv(ARQUIVO_QUERIES)
qrels = pd.read_csv(ARQUIVO_QRELS)

## 3. Carregar os embeddings

Funções genéricas para carregar e normalizar os embeddings do arquivo h5.

A ideia é que há apenas dois arquivos, um para os embeddings das questões e outro para os embeddings dos chunks.

O:.: Os embeddings da OpenAI já são normalizados (mas há uma perda na conversão de f32 pra f16 para salvar no H5), nem precisaria disso.

In [11]:
import h5py
import faiss
import numpy as np

def carregar_embeddings_do_arquivo(arquivo, coluna_id, nome_modelo):
    with h5py.File(arquivo, "r") as f:
        ids = f[coluna_id][:].astype(str)
        emb = f[nome_modelo][:].astype(np.float32)

    return ids, emb

def normalizar_embeddings(x):
    faiss.normalize_L2(x)
    return x

## 4. Criar índices FAISS

Função para criar um índice FAISS.

In [7]:
def criar_indice_faiss(nome_modelo):
    arquivo_docs = f'{PASTA_CADERNO_EMBEDDINGS}embeddings_docs_{nome_modelo}.h5'
    
    doc_keys, emb = carregar_embeddings_do_arquivo(arquivo_docs, 'DOC_KEY', nome_modelo)
    emb = normalizar_embeddings(emb)
    dim = emb.shape[1]
        
    index_faiss = faiss.IndexFlatL2(dim)
    index_faiss.add(emb)

    return doc_keys, index_faiss

Cria os índices FAISS para cada modelo.

Como serão testados mais de um modelo de embeddings, será criado um mapa de índice no seguinte formato:

<code>lista_doc_key = [....]</code>

<code>mapa_indice_faiss = {
   'nome_modelo_1': indice_faiss,
   'nome_modelo_...': indice_faiss,
   'nome_modelo_n': indice_faiss
}</code>

Obs.: Devido à forma como os embeddings foram criados, todos os DOC_KEY estão na mesma sequência. Por isso a lista de doc_key é separada do mapa.

In [12]:
from tqdm import tqdm

lista_doc_key = []
mapa_indice_faiss = {}
for nome_modelo in tqdm(MODELOS_DISPONIVEIS):
    lista_doc_key, indice = criar_indice_faiss(nome_modelo)
    mapa_indice_faiss[nome_modelo] = indice

100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:01<00:00,  1.69it/s]


## 5. Carregar embeddings das queries

Cria mapa dos embeddings das queries por modelo.

Como serão testados mais de um modelo de embeddings, será criado um mapa da seguinte forma:

<code>lista_query_keys = []</code>

<code>mapa_emb_queries = {
   'nome_modelo_1': lista_de_embeddings,
   'nome_modelo_...': lista_de_embeddings,
   'nome_modelo_n: lista_de_embeddings
}

Assim como para o mapa de índice, todos os QUERY_KEY estão na mesma sequência.

In [13]:
lista_query_keys = []
mapa_emb_queries = {}

for nome_modelo in tqdm(MODELOS_DISPONIVEIS):
    arquivo_queries = f'{PASTA_CADERNO_EMBEDDINGS}embeddings_queries_{nome_modelo}.h5'
    lista_query_keys, emb_queries = carregar_embeddings_do_arquivo(arquivo_queries, 'QUERY_KEY', nome_modelo)
    emb_queries = normalizar_embeddings(emb_queries)
    mapa_emb_queries[nome_modelo] = emb_queries

100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 77.63it/s]


## 6. Pesquisa as queries nos índices FAISS

In [33]:
from metricas import metricas

total_porcentagem = len(MODELOS_DISPONIVEIS)*len(queries)

def get_doc_key_score(distancias, indices_retornados):
    # Dados para retornar
    top_doc_keys = []
    scores = []

    # As distâncias e os índices retornados são no shape (1, k). Primeiro, transforma tudo em lista de tamanho k:
    distancias = list(distancias[0])
    indices_retornados = indices_retornados = list(indices_retornados[0])
      
    for d, i in zip(distancias, indices_retornados):
        top_doc_keys.append(lista_doc_key[i])
        scores.append(1 - d/2) # O score dessa forma é a similaridade de cosseno
        
    return top_doc_keys, scores

with tqdm(total=total_porcentagem) as pbar:
    for nome_modelo in MODELOS_DISPONIVEIS:              
        # Pega todos os embeddings das queries para o modelo em análise
        embeddings = mapa_emb_queries[nome_modelo]

        # Guarda a lista de resultados do modelo
        lista_resultados_modelo = []

        # Varre todas as queries da lista de queries
        for idx, id_query in enumerate(lista_query_keys):
            # Pega o embeddings da query
            emb_query = embeddings[idx:idx+1] # shape (1, dim)

            # Consulta no índice do modelo que está sendo avaliado
            n_docs = 1000
            distancias, indices_retornados = mapa_indice_faiss[nome_modelo].search(emb_query, 1000)
            # O resultado retornado já está na ordem correta.
            # Apenas converte os índices retornados para DOC_KEY e a distância para similaridade de cosseno.
            top_doc_keys, scores = get_doc_key_score(distancias, indices_retornados)
            
            # Adicionar ao df_resultados_pesquisas_semanticas 
            for rank, doc_key in enumerate(top_doc_keys, start=1):
                lista_resultados_modelo.append({
                    "QUERY_KEY": id_query,
                    "DOC_KEY": doc_key,
                    "RANK": rank
                })
            pbar.update(1)

        df_resultados = pd.DataFrame(lista_resultados_modelo)
        df_resultados["QUERY_KEY"] = df_resultados["QUERY_KEY"].astype(int)
        
        df_resultados.to_csv(f'{PASTA_RESULTADO_CADERNO}run_{nome_modelo}.csv', index=False)
        df_metricas = metricas(df_resultados, qrels, aproximacao_trec_eval=True, k=[10])
        print(f'\tnDCG@10 ({nome_modelo}): {df_metricas['nDCG@10'].mean():.4f}')

 48%|██████████████████████████████████████▋                                          | 66/138 [00:01<00:01, 61.72it/s]

	nDCG@10 (text-embedding-3-large): 0.2498


 84%|███████████████████████████████████████████████████████████████████▏            | 116/138 [00:01<00:00, 87.42it/s]

	nDCG@10 (text-embedding-3-small): 0.2838


100%|████████████████████████████████████████████████████████████████████████████████| 138/138 [00:01<00:00, 70.70it/s]

	nDCG@10 (gemini-embedding-001): 0.2914
